# 02 Baseline Model — Combined Dataset (Math + Portuguese)

Train a pass/fail classifier on the combined UCI student performance datasets. Steps: load combined data, split train/validation, build preprocessing (one-hot for categoricals, passthrough numeric), train baselines, evaluate, and persist the Random Forest pipeline.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

# Ensure project root on path
base_dir = Path("..").resolve()
sys.path.append(str(base_dir))

pd.set_option("display.max_columns", None)
processed_dir = base_dir / "data" / "processed"
models_dir = base_dir / "models"
models_dir.mkdir(exist_ok=True)


In [ ]:
# Load combined cleaned dataset
combined_path = processed_dir / "student_all_cleaned.csv"
df = pd.read_csv(combined_path)
print(df.shape)
df.head()


In [ ]:
# Separate features/target
X = df.drop(columns=["G3", "pass"])
y = df["pass"]

cat_cols = list(X.select_dtypes(include=["object", "string"]).columns)
num_cols = list(X.select_dtypes(exclude=["object", "string"]).columns)
cat_cols, num_cols

In [ ]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
# Preprocessing: one-hot encode categoricals, passthrough numerics
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

In [ ]:
# Baseline 1: Logistic Regression
log_reg = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(max_iter=500, class_weight="balanced"),
        ),
    ]
)
log_reg.fit(X_train, y_train)
log_pred = log_reg.predict(X_val)
print("Logistic Regression report:\n", classification_report(y_val, log_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, log_pred))

In [ ]:
# Baseline 2: Random Forest
rf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                random_state=42,
                class_weight="balanced",  # balance classes
                n_jobs=-1,
                max_depth=None,
            ),
        ),
    ]
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_val)
print("Random Forest report:\n", classification_report(y_val, rf_pred))
print("Confusion matrix:\n", confusion_matrix(y_val, rf_pred))

In [ ]:
# Choose a model to persist (random forest) and save
best_model = rf
model_path = models_dir / "pass_classifier_rf.joblib"
joblib.dump(best_model, model_path)
print(f"Saved model to {model_path}")


### Notes
- Dataset: combined math + Portuguese cleaned records (1044 rows).
- Random Forest currently performs best; tune hyperparameters or thresholds if recall needs to be higher.
- To switch to regression, set `y = df["G3"]`, remove class weights, and use a regressor with MAE/RMSE metrics.
